# T2.9 - Cohort-wide rank-1 attention distribution extraction

This notebook re-runs CLAM-TR v20 inference on the held-out test set of each fold to extract per-slide attention vectors, computes the rank-1 attention mass for each slide, and aggregates the cohort-wide distribution over the 9,128-slide noise-cleaned PANDA cohort.

**Output:**
- `rank1_attention_per_slide.csv` - one row per slide x (encoder, variant, hp, fold) with the rank-1 attention mass and the top-5 weights.
- `rank1_attention_distribution_summary.csv` - aggregate statistics (mean, median, P5, P95, fraction >= 0.99, fraction >= 0.999) per (encoder, variant, hp).

**Purpose:** Closes thesis review T2.9 (cohort-wide rank-1 attention distribution). The thesis currently reports rank-1 >= 0.9969 on 3 example slides only (Fig. 3.10-3.12); this notebook generalises that observation to all 9,128 slides across the 6 aggregator variants x 2 hp x 5 folds x 2 encoders factorial.

**Runtime:** approximately 4-8 hours on a Colab GPU (T4 or A100). The main cost is loading UNI / UNI2-h features (1024d / 1536d patch embeddings, ~9,128 slides) into GPU RAM once per encoder and running 120 forward sweeps over the held-out splits. Use `TEST_MODE = True` for a sanity check on fold 0 of ABMIL_bestHP / UNI before launching the full sweep.

**Resume:** the notebook writes per-`(encoder, variant, hp, fold)` partial CSVs to `PARTIAL_DIR` as it goes. If Colab disconnects, re-run from Cell 1 onward; completed chunks are skipped automatically.


## Setup: paths and dependencies

Assumes Mohamed's CLAM-TR v20 layout on Google Drive:
- `MyDrive/UNI_features/` - 9,128 `.pt` files (UNI v1, 1024d) keyed by slide ID.
- `MyDrive/UNI_PANDA_Features/` - 9,128 `.h5` files (UNI2-h, 1536d).
- `MyDrive/CLAM_TR_Results/v20_fullscale/{UNI,UNI2-h}/checkpoints/` - 60 `.pt` checkpoints per encoder (6 variants x 2 hp x 5 folds).
- `MyDrive/CLAM_TR_Results/v20_fullscale/{UNI,UNI2-h}/predictions/` - 60 `.json` per-fold prediction files containing the test-set slide IDs.

Outputs are written under `MyDrive/CLAM_TR_Results/v20_fullscale/rank1_attention/`.

**Edit `PROJECT_ROOT` and `RESULTS_ROOT` in Cell 3 if your Drive layout differs.**


In [ ]:
# T2.9 extraction step 1: mount Drive, set paths.
from google.colab import drive
drive.mount('/content/drive')

import os

# EDIT THIS to match your Drive layout.
DRIVE_ROOT = '/content/drive/MyDrive'

# Feature directories (one per encoder).
FEATURE_DIRS = {
    'UNI':    os.path.join(DRIVE_ROOT, 'UNI_features'),         # .pt files, 1024d
    'UNI2-h': os.path.join(DRIVE_ROOT, 'UNI_PANDA_Features'),   # .h5 files, 1536d
}
FILE_EXTS = {'UNI': '.pt', 'UNI2-h': '.h5'}
INPUT_DIMS = {'UNI': 1024, 'UNI2-h': 1536}

# CLAM-TR v20 results root (contains UNI/ and UNI2-h/ subdirs).
RESULTS_ROOT = os.path.join(DRIVE_ROOT, 'CLAM_TR_Results', 'v20_fullscale')

# T2.9 outputs go here.
T29_OUTPUT_DIR = os.path.join(RESULTS_ROOT, 'rank1_attention')
PARTIAL_DIR    = os.path.join(T29_OUTPUT_DIR, 'partial_per_fold')
os.makedirs(PARTIAL_DIR, exist_ok=True)

# Sanity check: PARTIAL_DIR exists + writeable (catches silent Drive mount issues).
assert os.path.isdir(PARTIAL_DIR), (
    f'PARTIAL_DIR does not exist after makedirs: {PARTIAL_DIR}. '
    f'Check DRIVE_ROOT is correct and writeable.')
_probe = os.path.join(PARTIAL_DIR, '.write_probe')
try:
    with open(_probe, 'w') as _f:
        _f.write('probe')
    os.remove(_probe)
except OSError as e:
    raise OSError(f'PARTIAL_DIR is not writeable: {PARTIAL_DIR}. Drive mount issue? {e}')


# Per-slide CSV (concatenation of all partial files).
PER_SLIDE_CSV = os.path.join(T29_OUTPUT_DIR, 'rank1_attention_per_slide.csv')
# Aggregate summary CSV.
SUMMARY_CSV   = os.path.join(T29_OUTPUT_DIR, 'rank1_attention_distribution_summary.csv')

# =====================================================
# TEST MODE — run only fold 0 of ABMIL_bestHP on UNI as
# a sanity check. Flip to False for the full sweep.
# =====================================================
TEST_MODE = True

print(f'DRIVE_ROOT   = {DRIVE_ROOT}')
print(f'RESULTS_ROOT = {RESULTS_ROOT}')
print(f'T29_OUTPUT   = {T29_OUTPUT_DIR}')
print(f'TEST_MODE    = {TEST_MODE}')


In [ ]:
# T2.9 extraction step 2: install Python dependencies.
# (torch + numpy + pandas are preinstalled on Colab; we add h5py and tqdm.)
import subprocess, sys
for pkg in ['h5py', 'tqdm']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)

import torch, numpy as np, pandas as pd, h5py, json, time
from tqdm.auto import tqdm
import random

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'torch={torch.__version__}, device={device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}, '
          f'{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

# Reproducibility (matches v20 training).
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True


## Model class definitions

All classes below are extracted verbatim from `CLAM-master/final-code/04-final-v20/CLAM_TR_v20_FullScale_Colab.ipynb` (cells 21, 23 of the v20 notebook). No structural changes; only the training-loop hooks are removed.

The training forward pass computes the gated-attention score `A` over all `N` patches, takes its softmax along `dim=0` (the patch dimension), and then for the dual-branch variants applies `torch.topk(A, K=128)` to select the top-K patches for the CLAM and retention branches. For T2.9 we record the **full softmax over all N patches** (BEFORE TopK selection) and report rank-1 mass = `max(A_softmax)`. This is the quantity the thesis interpretability section references (Eq. 2.3.5 / Fig. 3.10-3.12).


In [ ]:
# T2.9 extraction step 3: model classes extracted verbatim from
# CLAM_TR_v20_FullScale_Colab.ipynb cells 21 (components) + 23 (classes).
import torch
import torch.nn as nn
import torch.nn.functional as F


class GatedAttention(nn.Module):
    """v20 exact. Gated attention with temperature scaling."""
    def __init__(self, input_dim=512, hidden_dim=256, dropout=0.25):
        super().__init__()
        self.attention_a = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.Tanh(), nn.Dropout(dropout))
        self.attention_b = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.Sigmoid(), nn.Dropout(dropout))
        self.attention_c = nn.Linear(hidden_dim, 1)

    def forward(self, x, temperature=1.0):
        a = self.attention_a(x)
        b = self.attention_b(x)
        return self.attention_c(a * b).squeeze(-1) / temperature


class SwishGate(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.proj = nn.Linear(dim, dim)

    def forward(self, x):
        return x * torch.sigmoid(self.proj(x))


class FFNBlock(nn.Module):
    def __init__(self, dim, expansion=4, dropout=0.1):
        super().__init__()
        hidden = dim * expansion
        self.net = nn.Sequential(
            nn.Linear(dim, hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, dim), nn.Dropout(dropout))
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        return x + self.net(self.norm(x))


class RetentionHead(nn.Module):
    """Scaled dot-product attention (no spatial decay)."""
    def __init__(self, hidden_dim, num_heads):
        super().__init__()
        self.head_dim = hidden_dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.W_Q = nn.Linear(hidden_dim, self.head_dim, bias=False)
        self.W_K = nn.Linear(hidden_dim, self.head_dim, bias=False)
        self.W_V = nn.Linear(hidden_dim, self.head_dim, bias=False)

    def forward(self, x):
        Q, K, V = self.W_Q(x), self.W_K(x), self.W_V(x)
        scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        attn = F.softmax(scores, dim=-1)
        return torch.matmul(attn, V)


class SpatialRetentionHeadMult(nn.Module):
    """Multiplicative spatial decay: scores * D."""
    def __init__(self, hidden_dim, gamma, num_heads):
        super().__init__()
        self.head_dim = hidden_dim // num_heads
        self.gamma = gamma
        self.scale = self.head_dim ** -0.5
        self.W_Q = nn.Linear(hidden_dim, self.head_dim, bias=False)
        self.W_K = nn.Linear(hidden_dim, self.head_dim, bias=False)
        self.W_V = nn.Linear(hidden_dim, self.head_dim, bias=False)

    def forward(self, x, coords):
        Q, K, V = self.W_Q(x), self.W_K(x), self.W_V(x)
        diff = coords.unsqueeze(0) - coords.unsqueeze(1)
        distances = torch.norm(diff.float(), dim=-1)
        normalized = distances / (distances.max() + 1e-6)
        D = self.gamma ** normalized
        scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale * D
        attn = F.softmax(scores, dim=-1)
        return torch.matmul(attn, V)


class SpatialRetentionHeadAdd(nn.Module):
    """Additive spatial decay: scores + log(D)."""
    def __init__(self, hidden_dim, gamma, num_heads):
        super().__init__()
        self.head_dim = hidden_dim // num_heads
        self.gamma = gamma
        self.scale = self.head_dim ** -0.5
        self.W_Q = nn.Linear(hidden_dim, self.head_dim, bias=False)
        self.W_K = nn.Linear(hidden_dim, self.head_dim, bias=False)
        self.W_V = nn.Linear(hidden_dim, self.head_dim, bias=False)

    def forward(self, x, coords):
        Q, K, V = self.W_Q(x), self.W_K(x), self.W_V(x)
        diff = coords.unsqueeze(0) - coords.unsqueeze(1)
        distances = torch.norm(diff.float(), dim=-1)
        normalized = distances / (distances.max() + 1e-6)
        D = self.gamma ** normalized
        scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        scores = scores + torch.log(D + 1e-6)
        attn = F.softmax(scores, dim=-1)
        return torch.matmul(attn, V)


class SpatialRetentionMult(nn.Module):
    """Standalone spatial retention (multiplicative). NO dropout/norm/residual."""
    def __init__(self, hidden_dim=512, num_heads=8, gamma_values=None):
        super().__init__()
        assert gamma_values is not None and len(gamma_values) == num_heads
        self.heads = nn.ModuleList([
            SpatialRetentionHeadMult(hidden_dim, gamma_values[i], num_heads)
            for i in range(num_heads)])
        self.output_proj = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x, coords):
        head_outputs = [h(x, coords) for h in self.heads]
        return self.output_proj(torch.cat(head_outputs, dim=-1))


class SpatialRetentionAdd(nn.Module):
    def __init__(self, hidden_dim=512, num_heads=8, gamma_values=None):
        super().__init__()
        assert gamma_values is not None and len(gamma_values) == num_heads
        self.heads = nn.ModuleList([
            SpatialRetentionHeadAdd(hidden_dim, gamma_values[i], num_heads)
            for i in range(num_heads)])
        self.output_proj = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x, coords):
        head_outputs = [h(x, coords) for h in self.heads]
        return self.output_proj(torch.cat(head_outputs, dim=-1))


class RetentionBlock(nn.Module):
    """Enhanced retention with full component stack.
    Order: MHA -> output_proj -> dropout -> Norm -> Residual -> Gate -> FFN"""
    def __init__(self, hidden_dim=512, num_heads=8, gamma_values=None,
                 use_spatial=False, spatial_mode='multiplicative',
                 use_norm=True, norm_type='group', num_groups=8,
                 use_residual=True, use_gate=True, use_ffn=True,
                 ffn_expansion=4, dropout=0.25):
        super().__init__()
        self.use_spatial = use_spatial
        self.use_norm = use_norm
        self.use_residual = use_residual
        self.use_gate = use_gate
        self.use_ffn = use_ffn
        if use_spatial:
            assert gamma_values is not None and len(gamma_values) == num_heads
            HeadClass = (SpatialRetentionHeadMult if spatial_mode == 'multiplicative'
                         else SpatialRetentionHeadAdd)
            self.heads = nn.ModuleList([
                HeadClass(hidden_dim, gamma_values[i], num_heads)
                for i in range(num_heads)])
        else:
            self.heads = nn.ModuleList([
                RetentionHead(hidden_dim, num_heads) for _ in range(num_heads)])
        self.output_proj = nn.Linear(hidden_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)
        if use_norm:
            self.norm = (nn.GroupNorm(num_groups, hidden_dim)
                         if norm_type == 'group' else nn.LayerNorm(hidden_dim))
        if use_gate:
            self.gate = SwishGate(hidden_dim)
        if use_ffn:
            self.ffn = FFNBlock(hidden_dim, ffn_expansion, dropout)

    def forward(self, x, coords=None):
        identity = x
        if self.use_spatial:
            head_outputs = [h(x, coords) for h in self.heads]
        else:
            head_outputs = [h(x) for h in self.heads]
        out = self.output_proj(torch.cat(head_outputs, dim=-1))
        out = self.dropout(out)
        if self.use_norm:    out = self.norm(out)
        if self.use_residual: out = out + identity
        if self.use_gate:    out = self.gate(out)
        if self.use_ffn:     out = self.ffn(out)
        return out


class ABMIL(nn.Module):
    """Single-branch baseline. Uses ALL patches."""
    def __init__(self, input_dim, hidden_dim=512, n_classes=6, dropout=0.25):
        super().__init__()
        self.needs_coords = False
        self.feature_transform = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout))
        self.attention = GatedAttention(hidden_dim, hidden_dim // 2, dropout)
        self.classifier = nn.Linear(hidden_dim, n_classes)

    def forward(self, x, coords=None, temperature=1.0):
        h = self.feature_transform(x)
        A = F.softmax(self.attention(h, temperature), dim=0)
        M = torch.sum(A.unsqueeze(-1) * h, dim=0)
        return self.classifier(M)


class DualBranchModel(nn.Module):
    """Unified dual-branch model. Covers all non-ABMIL experiments."""
    def __init__(self, input_dim, hidden_dim=512, n_classes=6, dropout=0.25,
                 K=128, retention_module=None, needs_coords=False):
        super().__init__()
        self.K = K
        self.needs_coords = needs_coords
        self.feature_transform = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout))
        self.attention = GatedAttention(hidden_dim, hidden_dim // 2, dropout)
        self.retention_module = retention_module
        self.classifier = nn.Linear(hidden_dim, n_classes)

    def forward(self, x, coords=None, temperature=1.0):
        h = self.feature_transform(x)
        A = self.attention(h, temperature)
        A_softmax = F.softmax(A, dim=0)
        K_actual = min(self.K, h.size(0))
        topk_indices = torch.topk(A, K_actual).indices
        h_topk = h[topk_indices]
        A_topk = A_softmax[topk_indices]
        M_clam = torch.sum(A_topk.unsqueeze(-1) * h_topk, dim=0)
        if self.retention_module is None:
            M_retention = h_topk.mean(dim=0)
        elif self.needs_coords:
            coords_topk = coords[topk_indices]
            M_retention = self.retention_module(h_topk, coords_topk).mean(dim=0)
        else:
            M_retention = self.retention_module(h_topk).mean(dim=0)
        return self.classifier(M_clam + M_retention)


# Hyperparameter sets (verbatim from v20 cell 6).
HP_SETS = {
    'default': {
        'dropout': 0.25,
        'gamma_values': [0.80, 0.85, 0.90, 0.93, 0.95, 0.97, 0.98, 0.99],
    },
    'best': {
        'dropout': 0.15,
        'gamma_values': [0.88, 0.90, 0.92, 0.94, 0.96, 0.98, 0.99, 0.995],
    },
}

# Architecture constants (v20 CONFIG).
HIDDEN_DIM = 512
N_CLASSES  = 6
NUM_HEADS  = 8
K_TOPK     = 128
FFN_EXPANSION = 4
NUM_GROUPS = 8


def create_model(experiment_name, hp_name, input_dim):
    """Factory matching v20 cell 25. Returns nn.Module ready for state_dict load."""
    hp = HP_SETS[hp_name]
    dropout = hp['dropout']
    gamma_values = hp['gamma_values']
    if experiment_name.startswith('ABMIL'):
        return ABMIL(input_dim, HIDDEN_DIM, N_CLASSES, dropout)
    if experiment_name.startswith('DualBranch'):
        return DualBranchModel(input_dim, HIDDEN_DIM, N_CLASSES, dropout,
                               K_TOPK, retention_module=None, needs_coords=False)
    if experiment_name.startswith('RetNet'):
        retention = RetentionBlock(
            hidden_dim=HIDDEN_DIM, num_heads=NUM_HEADS, use_spatial=False,
            use_norm=True, norm_type='group', num_groups=NUM_GROUPS,
            use_residual=True, use_gate=True, use_ffn=True,
            ffn_expansion=FFN_EXPANSION, dropout=dropout)
        return DualBranchModel(input_dim, HIDDEN_DIM, N_CLASSES, dropout,
                               K_TOPK, retention_module=retention, needs_coords=False)
    if experiment_name.startswith('SpatialMult_') and 'RetNet' not in experiment_name:
        spatial = SpatialRetentionMult(
            hidden_dim=HIDDEN_DIM, num_heads=NUM_HEADS, gamma_values=gamma_values)
        return DualBranchModel(input_dim, HIDDEN_DIM, N_CLASSES, dropout,
                               K_TOPK, retention_module=spatial, needs_coords=True)
    if experiment_name.startswith('SpatialMultRetNet'):
        retention = RetentionBlock(
            hidden_dim=HIDDEN_DIM, num_heads=NUM_HEADS, gamma_values=gamma_values,
            use_spatial=True, spatial_mode='multiplicative',
            use_norm=True, norm_type='group', num_groups=NUM_GROUPS,
            use_residual=True, use_gate=True, use_ffn=True,
            ffn_expansion=FFN_EXPANSION, dropout=dropout)
        return DualBranchModel(input_dim, HIDDEN_DIM, N_CLASSES, dropout,
                               K_TOPK, retention_module=retention, needs_coords=True)
    if experiment_name.startswith('SpatialAdd'):
        spatial = SpatialRetentionAdd(
            hidden_dim=HIDDEN_DIM, num_heads=NUM_HEADS, gamma_values=gamma_values)
        return DualBranchModel(input_dim, HIDDEN_DIM, N_CLASSES, dropout,
                               K_TOPK, retention_module=spatial, needs_coords=True)
    raise ValueError(f'Unknown experiment: {experiment_name}')

print('Model classes + factory defined.')


## Data loader

Per-fold test-set slide IDs are read from the existing prediction files `{RESULTS_ROOT}/{encoder}/predictions/{variant}_{hp}_fold{N}_predictions.json`. Each file contains `{slide_ids: [...], predictions: [...], labels: [...]}` with 1,803 slides for the 1/5 hold-out split (matches the dataset loader in v20 notebook cell 19). This lets us reproduce the exact test partition the model was scored on without re-loading the Kaggle 5-fold CSV.

Feature tensors are loaded on demand (one slide at a time) to keep peak RAM low. For the full sweep this is ~1.8 GB of `.pt` IO per fold for UNI and ~2.7 GB of `.h5` IO for UNI2-h. We do NOT preload all 9,128 slides into RAM as the training notebook does, because inference is sequential and the read cost is dominated by Drive IO rather than GPU compute.


In [ ]:
# T2.9 extraction step 4: data loader (matches v20 cell 19 _load_one).
import os, json, h5py
import numpy as np
import torch


def load_slide_features(slide_id, features_dir, file_ext):
    """Load one slide's (features, coords) tensors. Matches v20 PANDADataset._load_one.

    Returns:
        features: float32 tensor [N_patches, input_dim]
        coords:   float32 tensor [N_patches, 2]
    Returns (None, None) if the file is missing or malformed.
    """
    fpath = os.path.join(features_dir, f'{slide_id}{file_ext}')
    if not os.path.exists(fpath):
        return None, None
    try:
        if file_ext == '.pt':
            data = torch.load(fpath, map_location='cpu', weights_only=False)
            if isinstance(data, dict):
                features = data.get('features',
                                    data.get('embeddings', list(data.values())[0]))
                features = (torch.tensor(features, dtype=torch.float32)
                            if isinstance(features, np.ndarray) else features.float())
                if features.ndim == 3:
                    features = features.squeeze(0)
                coords_raw = data.get('coords', data.get('coordinates', None))
                if coords_raw is None:
                    coords = torch.zeros(features.shape[0], 2, dtype=torch.float32)
                else:
                    coords = (torch.tensor(coords_raw, dtype=torch.float32)
                              if isinstance(coords_raw, np.ndarray)
                              else coords_raw.float())
                    if coords.ndim == 3:
                        coords = coords.squeeze(0)
            elif isinstance(data, torch.Tensor):
                features = data.float()
                if features.ndim == 3:
                    features = features.squeeze(0)
                coords = torch.zeros(features.shape[0], 2, dtype=torch.float32)
        elif file_ext == '.h5':
            with h5py.File(fpath, 'r') as f:
                features = torch.tensor(f['features'][:], dtype=torch.float32).squeeze(0)
                coords   = torch.tensor(f['coords'][:],   dtype=torch.float32).squeeze(0)
        else:
            raise ValueError(f'Unknown extension: {file_ext}')
    except Exception as e:
        print(f'  WARN: failed to load {slide_id}: {e}')
        return None, None
    return features, coords


def load_test_slide_ids(encoder, variant, hp, fold):
    """Return list of slide IDs in the held-out test set for this fold.
    Reads {RESULTS_ROOT}/{encoder}/predictions/{variant}_{hp}HP_fold{fold}_predictions.json.
    """
    pred_path = os.path.join(
        RESULTS_ROOT, encoder, 'predictions',
        f'{variant}_{hp}HP_fold{fold}_predictions.json')
    if not os.path.exists(pred_path):
        return None, pred_path
    with open(pred_path) as f:
        pred = json.load(f)
    return pred['slide_ids'], pred_path


print('Data loader defined.')


## Attention extraction

For each slide the function below runs a single forward pass to the point where the gated-attention vector `A` is computed over all `N` patches, applies `F.softmax(A, dim=0)` (the same softmax Eq. 2.3.5 in the thesis uses), and records:

- `n_patches` - the number of patches `N`.
- `rank1_mass` - `max(A_softmax)`, i.e. the attention mass on the single most-attended patch.
- `rank2_mass ... rank5_mass` - the next four largest weights, for the optional rank-k tail summary.
- `topk_idx_rank1` - the patch index that received `rank1_mass`.

Important: the score is computed BEFORE the TopK selection step `torch.topk(A, K=128)` that the dual-branch variants apply. This matches the quantity reported in Fig. 3.10-3.12 of the thesis (rank-1 = 0.9969 / 0.9994 / 0.9998 on the three example slides).


In [ ]:
# T2.9 extraction step 5: per-slide attention extraction.
import torch
import torch.nn.functional as F


@torch.no_grad()
def extract_attention(model, features, coords, top_k=5):
    """Run a single slide through `model` and return the rank-1 attention mass + top-K weights.

    Reproduces the v20 forward path up to the softmax over the gated-attention
    scores (BEFORE TopK selection). Works for ABMIL and DualBranchModel.

    Args:
        model:    eval()-mode nn.Module (ABMIL or DualBranchModel)
        features: float tensor [N, input_dim] on device
        coords:   float tensor [N, 2] on device (unused by ABMIL)
        top_k:    number of top weights to record (default 5)

    Returns dict with keys: n_patches, rank1_mass, rank{2..top_k}_mass, top1_idx,
    sum_top_k (mass on the union of the top_k patches).
    """
    # Both ABMIL and DualBranchModel have feature_transform + attention.
    h = model.feature_transform(features)
    # Use temperature=1.0 — matches inference path in v20 cell 35 _run_single_fold val loop.
    raw_scores = model.attention(h, 1.0)              # shape [N]
    A_softmax  = F.softmax(raw_scores, dim=0)         # shape [N]
    n = A_softmax.shape[0]
    k = min(top_k, n)
    top_vals, top_idx = torch.topk(A_softmax, k)
    out = {
        'n_patches': int(n),
        'top1_idx':  int(top_idx[0].item()),
        'sum_top_k': float(top_vals.sum().item()),
    }
    for r in range(1, top_k + 1):
        out[f'rank{r}_mass'] = float(top_vals[r-1].item()) if r <= k else float('nan')
    return out


def load_checkpoint(ckpt_path, model, device):
    """Load a v20 checkpoint (.pt) into `model`. Tolerates several state_dict layouts."""
    state = torch.load(ckpt_path, map_location=device, weights_only=False)
    if isinstance(state, dict) and 'model_state_dict' in state:
        sd = state['model_state_dict']
    elif isinstance(state, dict) and 'state_dict' in state:
        sd = state['state_dict']
    elif isinstance(state, dict) and all(isinstance(v, torch.Tensor) for v in state.values()):
        sd = state
    else:
        raise ValueError(f'Unrecognised checkpoint format at {ckpt_path}: '
                         f'top-level keys = {list(state.keys()) if isinstance(state, dict) else type(state)}')
    missing, unexpected = model.load_state_dict(sd, strict=False)
    if missing:
        print(f'  WARN: missing keys in checkpoint: {missing[:5]}...')
    if unexpected:
        print(f'  WARN: unexpected keys in checkpoint: {unexpected[:5]}...')
    return model

print('Attention extraction defined.')


## Main aggregation loop

Loops over all `(encoder, variant, hp, fold)` combinations, loads the checkpoint + test-set slide IDs, runs `extract_attention` on each slide, and writes a per-fold partial CSV under `PARTIAL_DIR`. On resume, completed (encoder, variant, hp, fold) chunks are skipped.

Full sweep size: 2 encoders x 6 variants x 2 hp x 5 folds = 120 model instances x ~1,803 slides each = ~216,360 inferences. Each inference is 1 ms - 30 ms depending on N_patches and aggregator type, so total wall time is dominated by Drive feature IO (estimated 4-8 hours).


In [ ]:
# T2.9 extraction step 6: main loop. Writes per-fold partial CSVs as it goes.
import os, time
import pandas as pd
import torch
from tqdm.auto import tqdm

VARIANTS = ['ABMIL', 'DualBranch', 'RetNet',
            'SpatialMult', 'SpatialMultRetNet', 'SpatialAdd']
HPS      = ['default', 'best']
FOLDS    = list(range(5))

if TEST_MODE:
    ENCODERS_TO_RUN = ['UNI']
    VARIANTS_TO_RUN = ['ABMIL']
    HPS_TO_RUN      = ['best']
    FOLDS_TO_RUN    = [0]
    print('TEST MODE: running ABMIL_bestHP / UNI / fold 0 only.')
else:
    ENCODERS_TO_RUN = ['UNI', 'UNI2-h']
    VARIANTS_TO_RUN = VARIANTS
    HPS_TO_RUN      = HPS
    FOLDS_TO_RUN    = FOLDS
    print(f'FULL SWEEP: {len(ENCODERS_TO_RUN)}x{len(VARIANTS_TO_RUN)}x'
          f'{len(HPS_TO_RUN)}x{len(FOLDS_TO_RUN)} = '
          f'{len(ENCODERS_TO_RUN)*len(VARIANTS_TO_RUN)*len(HPS_TO_RUN)*len(FOLDS_TO_RUN)} '
          'checkpoints x ~1,803 slides each.')


def partial_csv_path(encoder, variant, hp, fold):
    return os.path.join(PARTIAL_DIR,
                        f'{encoder}_{variant}_{hp}HP_fold{fold}.csv')


def process_fold(encoder, variant, hp, fold):
    """Process one (encoder, variant, hp, fold) chunk; write partial CSV; return rows."""
    out_path = partial_csv_path(encoder, variant, hp, fold)
    if os.path.exists(out_path):
        # Resume: skip already-completed chunks.
        return pd.read_csv(out_path)

    # 1. Resolve checkpoint.
    ckpt_name = f'{variant}_{hp}HP_fold{fold}_epoch.pt'
    ckpt_path = os.path.join(RESULTS_ROOT, encoder, 'checkpoints', ckpt_name)
    if not os.path.exists(ckpt_path):
        print(f'  SKIP: missing checkpoint {ckpt_path}')
        return None

    # 2. Resolve test slide IDs from predictions JSON.
    slide_ids, pred_path = load_test_slide_ids(encoder, variant, hp, fold)
    if slide_ids is None:
        print(f'  SKIP: missing predictions file {pred_path}')
        return None

    # 3. Build model + load checkpoint.
    exp_name  = f'{variant}_{hp}HP'
    input_dim = INPUT_DIMS[encoder]
    model = create_model(exp_name, hp, input_dim).to(device).eval()
    model = load_checkpoint(ckpt_path, model, device)

    # 4. Inference.
    features_dir = FEATURE_DIRS[encoder]
    file_ext     = FILE_EXTS[encoder]
    rows = []
    t0 = time.time()
    for sid in tqdm(slide_ids,
                    desc=f'{encoder}/{variant}_{hp}HP/fold{fold}',
                    leave=False):
        features, coords = load_slide_features(sid, features_dir, file_ext)
        if features is None:
            rows.append({
                'encoder': encoder, 'variant': variant, 'hp': hp, 'fold': fold,
                'slide_id': sid, 'n_patches': 0, 'rank1_mass': float('nan'),
            })
            continue
        features = features.to(device, non_blocking=True)
        coords   = coords.to(device, non_blocking=True)
        info = extract_attention(model, features, coords, top_k=5)
        info.update({'encoder': encoder, 'variant': variant, 'hp': hp,
                     'fold': fold, 'slide_id': sid})
        rows.append(info)

    # 5. Free GPU.
    del model
    if device.type == 'cuda':
        torch.cuda.empty_cache()

    # 6. Persist partial CSV.
    df = pd.DataFrame(rows)
    df.to_csv(out_path, index=False)
    elapsed = time.time() - t0
    print(f'  DONE {encoder}/{variant}_{hp}HP/fold{fold}: '
          f'{len(rows)} slides in {elapsed:.1f}s '
          f'(mean rank1={df["rank1_mass"].mean():.4f}, '
          f'median={df["rank1_mass"].median():.4f})')
    return df


# ----- Main loop -----
all_dfs = []
for encoder in ENCODERS_TO_RUN:
    for variant in VARIANTS_TO_RUN:
        for hp in HPS_TO_RUN:
            for fold in FOLDS_TO_RUN:
                df = process_fold(encoder, variant, hp, fold)
                if df is not None:
                    all_dfs.append(df)

if all_dfs:
    combined = pd.concat(all_dfs, ignore_index=True)
    combined.to_csv(PER_SLIDE_CSV, index=False)
    print(f'\nWrote {len(combined):,} rows to {PER_SLIDE_CSV}')
else:
    print('No chunks processed — check paths above.')


## Cohort-wide aggregation

Pools the per-slide rank-1 attention masses across the 5 folds (a complete 1/5 + 4/5 + ... + 5/5 = 5/5 cohort partition per (encoder, variant, hp)) and reports mean, median, P5, P95, and the fraction of slides with rank-1 mass >= 0.99 / >= 0.999.


In [ ]:
# T2.9 extraction step 7: cohort-wide aggregation.
import os
import numpy as np
import pandas as pd

# Reload from disk (works after restart, or after combine in cell 12).
if not os.path.exists(PER_SLIDE_CSV):
    # Rebuild from partial files if main loop wrote partials but not the combined csv.
    # Defensive: ensure PARTIAL_DIR exists before listdir.
    os.makedirs(PARTIAL_DIR, exist_ok=True)
    files = [f for f in sorted(os.listdir(PARTIAL_DIR)) if f.endswith('.csv')]
    if not files:
        raise FileNotFoundError(
            f'No partial CSVs found in {PARTIAL_DIR}. '
            f'Did Cell 12 (main loop) run successfully? '
            f'If TEST_MODE=True, check that ABMIL_bestHP UNI fold 0 inference produced a partial CSV. '
            f'You can also re-run Cell 12 after fixing the upstream error.')
    parts = [pd.read_csv(os.path.join(PARTIAL_DIR, f)) for f in files]
    combined = pd.concat(parts, ignore_index=True)
    combined.to_csv(PER_SLIDE_CSV, index=False)
else:
    combined = pd.read_csv(PER_SLIDE_CSV)
print(f'Loaded {len(combined):,} rows from {PER_SLIDE_CSV}.')

# Drop slides whose features failed to load (n_patches = 0 / rank1 = NaN).
valid = combined.dropna(subset=['rank1_mass']).copy()
n_dropped = len(combined) - len(valid)
if n_dropped:
    print(f'  Dropped {n_dropped} slides with missing features.')


def summarise(group):
    r = group['rank1_mass'].to_numpy()
    return pd.Series({
        'n_slides':         int(len(r)),
        'mean_rank1':       float(np.mean(r)),
        'median_rank1':     float(np.median(r)),
        'std_rank1':        float(np.std(r, ddof=1)) if len(r) > 1 else float('nan'),
        'p5_rank1':         float(np.percentile(r, 5)),
        'p25_rank1':        float(np.percentile(r, 25)),
        'p75_rank1':        float(np.percentile(r, 75)),
        'p95_rank1':        float(np.percentile(r, 95)),
        'min_rank1':        float(np.min(r)),
        'max_rank1':        float(np.max(r)),
        'frac_ge_0.50':     float(np.mean(r >= 0.50)),
        'frac_ge_0.90':     float(np.mean(r >= 0.90)),
        'frac_ge_0.99':     float(np.mean(r >= 0.99)),
        'frac_ge_0.999':    float(np.mean(r >= 0.999)),
        'frac_ge_0.9969':   float(np.mean(r >= 0.9969)),  # thesis anchor (Fig. 3.12 G5)
        'mean_n_patches':   float(group['n_patches'].mean()),
        'median_n_patches': float(group['n_patches'].median()),
    })


summary = (valid.groupby(['encoder', 'variant', 'hp'], dropna=False)
                .apply(summarise)
                .reset_index())
summary.to_csv(SUMMARY_CSV, index=False)
print(f'Wrote summary to {SUMMARY_CSV}')
summary


## Output for thesis

Prints the anchor sentence that can be pasted into Ch.3 Section 3.x (Interpretability) of the thesis to replace the current 3-example-slide claim with a cohort-wide statement. Run this cell after the main loop completes (or once enough chunks have been processed for the configurations the thesis cites).


In [ ]:
# T2.9 extraction step 8: thesis-ready output strings.
# Generates anchor sentences for Ch.3 interpretability section.
import pandas as pd

if 'summary' not in dir():
    summary = pd.read_csv(SUMMARY_CSV)


def row(enc, var, hp):
    m = ((summary['encoder'] == enc)
         & (summary['variant'] == var)
         & (summary['hp']      == hp))
    return summary[m].iloc[0] if m.any() else None


def anchor(enc, var, hp):
    r = row(enc, var, hp)
    if r is None:
        return f'  {enc}/{var}_{hp}HP: [not yet processed]'
    return (
        f"  {enc}/{var}_{hp}HP across {int(r['n_slides']):,} slides: "
        f"rank-1 attention mass mean = {r['mean_rank1']:.4f}, "
        f"median = {r['median_rank1']:.4f}, "
        f"P5 / P95 = {r['p5_rank1']:.4f} / {r['p95_rank1']:.4f}, "
        f"min / max = {r['min_rank1']:.4f} / {r['max_rank1']:.4f}, "
        f"frac(rank-1 >= 0.99) = {r['frac_ge_0.99']*100:.1f}%, "
        f"frac(rank-1 >= 0.999) = {r['frac_ge_0.999']*100:.1f}%, "
        f"frac(rank-1 >= 0.9969) = {r['frac_ge_0.9969']*100:.1f}%."
    )


print('=' * 80)
print('T2.9 ANCHOR SENTENCES — paste into Ch.3 interpretability section')
print('=' * 80)
for enc in ['UNI', 'UNI2-h']:
    for hp in ['best', 'default']:
        print(f'\n[{enc} / {hp}HP]')
        for var in ['ABMIL', 'DualBranch', 'RetNet',
                    'SpatialMult', 'SpatialMultRetNet', 'SpatialAdd']:
            print(anchor(enc, var, hp))

# Headline sentence for the thesis-cited configuration
# (UNI2-h x SpatialMultRetNet x best — the one Fig. 3.12 was rendered from).
print('\n' + '=' * 80)
print('HEADLINE — paste into Ch.3 §sec:interpretability')
print('=' * 80)
headline_row = row('UNI2-h', 'SpatialMultRetNet', 'best')
if headline_row is not None:
    print(
        f"On the SpatialMultRetNet (best HP) x UNI2-h configuration "
        f"aggregated across {int(headline_row['n_slides']):,} held-out test "
        f"slides (5 folds combined), the rank-1 attention mass has "
        f"mean = {headline_row['mean_rank1']:.4f}, "
        f"median = {headline_row['median_rank1']:.4f}, "
        f"with 5th/95th percentiles {headline_row['p5_rank1']:.4f} / "
        f"{headline_row['p95_rank1']:.4f}; "
        f"{headline_row['frac_ge_0.99']*100:.1f}% of slides exhibit "
        f"rank-1 >= 0.99, and {headline_row['frac_ge_0.999']*100:.1f}% "
        f"exhibit rank-1 >= 0.999, confirming that the three example "
        f"slides shown in Figs. 3.10-3.12 (rank-1 = 0.9998 / 0.9994 / "
        f"0.9969) are representative of cohort-wide attention concentration."
    )
else:
    print('  Headline configuration (UNI2-h / SpatialMultRetNet / bestHP) not yet processed.')
